In [ ]:
#This needs to be added in the main file just below where we get filtered_ds
save_path = "clinical_trial_simplification_filtered"
filtered_ds.save_to_disk(save_path)

print(f"Saved filtered dataset to: {save_path}")

In [ ]:
# loading from disk
from datasets import load_from_disk
filtered_ds = load_from_disk("clinical_trial_simplification_filtered")

def build_inputs(example):
    example["input_text"] = (
        "Summarize this clinical trial for a patient:\n\n" +
        example["detailed_description"]
    )
    return example

filtered_ds = filtered_ds.map(build_inputs)

filtered_ds[0]["input_text"], filtered_ds[0]["brief_summary"]


# chosen model - FLAN-T5-Large

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-large"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)


In [ ]:
#Stop-word removal does not help transformer models (they learn context), but light normalization does.
#So here strip HTML, normalize whitespace, optionally remove stopwords only from input (not target summaries), lowercase (optional)


#We can decide later whether we need this entire block of Stop-word Removal / Preprocessing

#light preprocessing
import re
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def clean_text(text):
    # Remove HTML tags
    text = re.sub(r"<.*?>", " ", text)
    # Remove weird characters
    text = re.sub(r"[\r\n\t]+", " ", text)
    # Collapse excess spaces
    text = re.sub(r"\s+", " ", text).strip()
    return text

def preprocess(example):
    text = example["detailed_description"]
    text = clean_text(text)

    # OPTIONAL: stopword removal (not for target)
    tokens = text.split()
    tokens = [t for t in tokens if t.lower() not in stop_words]
    text_no_stop = " ".join(tokens)

    example["cleaned_input"] = (
        "Summarize this clinical trial for a patient:\n\n" + text_no_stop
    )

    example["cleaned_target"] = clean_text(example["brief_summary"])
    return example

filtered_ds = filtered_ds.map(preprocess)


In [ ]:
# We will use the cleaned fields, cleaned_input, cleaned_target
max_input_len = 1024
max_target_len = 256

def tokenize_batch(batch):
    model_inputs = tokenizer(
        batch["cleaned_input"],
        max_length=max_input_len,
        padding="max_length",
        truncation=True,
    )

    labels = tokenizer(
        batch["cleaned_target"],
        max_length=max_target_len,
        padding="max_length",
        truncation=True,
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_ds = filtered_ds.map(
    tokenize_batch,
    batched=True,
    remove_columns=filtered_ds.column_names
)


In [ ]:
#Split train / validation

tokenized_ds = tokenized_ds.train_test_split(test_size=0.1)
train_ds = tokenized_ds["train"]
val_ds = tokenized_ds["test"]


In [ ]:
# Define Loss Function

loss = CrossEntropyLoss(ignore_index=-100)

#Transformers automatically use cross-entropy loss over decoder tokens


# We don’t need to define it manually unless customizing. But if we want an explicit loss override, we can subclass the model. For now, we use built-in loss (best practice for seq2seq).


In [ ]:
# Fine-Tune the Model

from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# Training arguments

from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./ct_summary_model",
    predict_with_generate=True,
    evaluation_strategy="epoch",
    per_device_train_batch_size=2,   # adjust if GPU available
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,   # effective batch size = 8
    learning_rate=2e-5,
    num_train_epochs=4,
    warmup_steps=200,
    logging_steps=50,
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,            # if GPU supports it
    report_to="none",     # disable HF logging
)


# Creating trainer and training below
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    tokenizer=tokenizer,
)
trainer.train()


In [ ]:

#After training, run inference:
def simplify(text):
    inp = "Summarize this clinical trial for a patient:\n\n" + text
    tokens = tokenizer(
        inp, return_tensors="pt", truncation=True, max_length=1024
    )
    output = model.generate(**tokens, max_length=256)
    return tokenizer.decode(output[0], skip_special_tokens=True)

print(simplify("Your clinical trial description here..."))
